# Few-Shot BAT Classification (Pipeline 1)

Local Jupyter notebook version, adapted from `phase2a_local.py`.

**What changed vs. the desktop Groq script:**
- LLM backend: **Kimi K2.5 via the UVA RC GenAI endpoint** (not Groq), called with raw `httpx` (SSE-aware), not the OpenAI/Groq SDK.
- **Few-shot** prompting: 10 human-labeled examples (`few_shot_ex10.csv`, Nadia's labels) are embedded in the system prompt ahead of each post to classify.
- **No concurrency / no Slurm array / no batch flushing** — this is a 30-row job, so it just loops sequentially with a small delay between calls and saves after every row (cheap, resume-safe, no need for batching machinery).
- Posts-only (comments logic from the original script is dropped — you weren't using it anyway).

**Inputs**
- `few_shot_ex10.csv` — 10 posts with Nadia's ground-truth EX/EMO/COG/MD labels (few-shot examples)
- `merged_post30_empty.csv` — 30 posts to classify (EX/EMO/COG/MD/bat_score/*_reasoning columns are blank)

**Output**
- `few_shot_bat_classification_labeled.csv` — the 30 posts with LLM-predicted EX/EMO/COG/MD/bat_score/reasoning filled in

Run cells top to bottom. Re-running is safe — already-classified `post_id`s are skipped on re-run (checkpointing).

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
FEWSHOT_FILE = "few_shot_ex10.csv"
INPUT_FILE   = "merged_post30_empty.csv"
OUTPUT_FILE  = "few_shot_bat_classification_labeled.csv"

MODEL      = "Kimi K2.5"     # confirmed exact model id via GET /api/models
MAX_TOKENS = 4000            # reasoning models on this endpoint need >=4000, per prior runs
TEMPERATURE = 0.0            # deterministic, matches phase2a_local.py convention
DELAY_SECONDS = 1.0          # small pause between calls — no concurrency needed at n=30

import os
print(f"Fewshot file : {FEWSHOT_FILE}")
print(f"Input file   : {INPUT_FILE}")
print(f"Output file  : {OUTPUT_FILE}")
print(f"Model        : {MODEL}")

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────
import httpx
import pandas as pd
import json
import time
import os
import re
import sys

In [ ]:
# ── Load .env (same convention as phase2a_local.py) ──────────────────────
def _load_env_file():
    for candidate in [os.path.join(os.getcwd(), ".env")]:
        if os.path.exists(candidate):
            print(f"Loading .env from: {candidate}")
            with open(candidate) as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    line = re.sub(r"^export\s+", "", line)
                    if "=" in line:
                        k, _, v = line.partition("=")
                        k = k.strip()
                        v = v.strip().strip('"').strip("'")
                        if k and k not in os.environ:
                            os.environ[k] = v
            return candidate
    return None

_found = _load_env_file()
if not _found:
    print("(No .env file found in cwd — falling back to shell environment)")

UVARC_API_KEY = os.environ.get("UVARC_GenAI_API")
if not UVARC_API_KEY:
    sys.exit(
        "\nERROR: UVARC_GenAI_API not found.\n"
        "  Put it in a .env file in the same folder as this notebook:\n"
        "    UVARC_GenAI_API=your_key_here\n"
    )

BASE_URL = "https://open-webui.rc.virginia.edu/api"
CHAT_URL = f"{BASE_URL}/chat/completions"
print("API key loaded, endpoint:", CHAT_URL)

In [ ]:
# ── BAT rubric (identical to phase2a_local.py — do not edit lightly, this
# is the validated rubric used for Phase 2A) ──────────────────────────────
BAT_INSTRUCTIONS = """You are a researcher applying the Burnout Assessment Tool (BAT) to Reddit text.
Your job is to decide YES or NO for each of four burnout dimensions.  We will consider any form of burnout
and stress signal written in any tense such as past, present and future. For example: "I will feel stress..",
"I have gone through a lot of stress or I was stressed or burnout", "i am having sleep trouble or having
trouble balancing my work and personal life".

IMPORTANT RULES BEFORE YOU START:
- Read the text cold, with no assumptions about whether burnout is present.
- This is a SENSITIVITY-FIRST task. When in doubt, lean YES.
  Missing a true burnout signal is worse than flagging a stress-adjacent one.
- A post asking others about their experience is NOT the same as expressing it yourself.
- Naturalistic Reddit language rarely uses clinical terms — look for the meaning, not exact words.
- The word "burnout" alone with no other signal = NO. Any supporting signal alongside it = YES.

\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501
EX \u2014 Exhaustion
\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501
Definition: Energy loss from work \u2014 physical (tiredness, feeling weak) AND/OR mental
(feeling drained, worn-out). Includes sustained overload that implies depletion even
without the exact word "drained".

YES if any of:
  - Explicit depletion: "drained", "nothing left", "used up", "running on empty",
    "physically broken", "can\u2019t decompress", "sleep doesn\u2019t help"
  - Sustained overload personally described over weeks or months:
    "tough couple of years", "impossible deadlines", "never ending [workload]",
    "always something left to do", chronic on-call or work pressure as personal cost
  - Physical or health deterioration attributed to work:
    "health has been deteriorating [since this job]", "getting sick from work"
  - Persistent misery tied to the job: "been miserable since [starting this role]"
  - Lack of energy to start work, feeling completely used up after working

NO if:
  - A single mentioned bad day with no sustained element
  - Asking others if they experience exhaustion (not expressing it personally)
  - Only boredom or dissatisfaction with no energy or health cost described

\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501
EMO \u2014 Emotional Impairment
\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501
Definition: Intense, persistent, or disproportionate emotional reactions tied to work.
Does NOT require explicit "loss of control" \u2014 strong sustained negative emotion qualifies.

YES if any of:
  - Strong hate or intense aversion toward the work situation:
    "I HATE [job/role/situation]", "this job is making me miserable"
  - Repeated or stacked emotional signals in the same post indicating sustained distress
    (e.g., "punch in the face" used twice, or multiple frustration phrases together)
  - Snapping, crying unexpectedly, or overreacting at work
  - Feeling upset, sad, or angry without a clear single cause
  - Persistent irritability or frustration tied to work beyond one incident
  - Feeling frustrated and angry at work, feeling upset or sad without knowing why
  - Feeling unable to control one\u2019s emotions at work, irritability, overreacting

NO if:
  - A single proportionate frustration about one event mentioned once and calmly
  - Mild annoyance described without intensity or repetition
  - Asking others if they feel frustrated (not expressing it personally)

\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501
COG \u2014 Cognitive Impairment
\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501
Definition: Difficulty with memory, focus, or decision-making at work.
Includes feeling cognitively overwhelmed by job demands (volume, complexity, pace).

YES if any of:
  - Feeling overwhelmed by cognitive demands: volume of alerts, tasks, or complexity
    (even in a newer role, if the overwhelm goes beyond normal new-job adjustment)
  - Brain fog, forgetting procedures or tasks, trouble concentrating
  - Indecision or inability to make decisions that would normally be easy
  - Difficulty learning or keeping up with what the job demands
  - Being absent-minded, forgetful, or mentally scattered at work
  - Difficulties thinking clearly, poor memory, attention or concentration at work

NO if:
  - Brand new to role AND describes normal learning difficulty with no signs of distress
  - Asking others about cognitive difficulty without expressing it personally
  - Feeling generally confused with no work-specific cognitive symptom

\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501
MD \u2014 Mental Distance
\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501
Definition: Persistent psychological withdrawal \u2014 indifference, cynicism, aversion, autopilot.

YES if any of:
  - Explicit loss of meaning or interest: "what\u2019s the point", "don\u2019t care anymore",
    "I used to love this but now feel nothing", "no longer want to"
  - Going through the motions or autopilot described personally
  - Active avoidance of work tasks or colleagues
  - Persistent cynical or resentful tone throughout the post
  - Persistent dread of work: "I dread going in / this role / these tasks"
  - Wanting to escape driven by disengagement rather than career ambition
  - Withdrawing mentally or physically from work, avoiding contact with colleagues

NO if:
  - Asking others about engagement (not expressing own detachment)
  - Single bad day or one-off complaint
  - Considering career change out of ambition or curiosity (active agency, not withdrawal)
  - Mild boredom mentioned once without sustained withdrawal signals

\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501

Respond with JSON only. No markdown fences. No explanation outside the JSON.

For each category provide a reasoning field explaining the decision whether YES or NO:
  - If YES: quote the exact phrase from the text that triggered YES.
  - If NO:  write one sentence explaining why the text did not meet the threshold.

{
  "EX":          "YES" or "NO",
  "EMO":         "YES" or "NO",
  "COG":         "YES" or "NO",
  "MD":          "YES" or "NO",
  "EX_reasoning":  "quoted phrase if YES  /  one-sentence explanation if NO",
  "EMO_reasoning": "quoted phrase if YES  /  one-sentence explanation if NO",
  "COG_reasoning": "quoted phrase if YES  /  one-sentence explanation if NO",
  "MD_reasoning":  "quoted phrase if YES  /  one-sentence explanation if NO"
}"""

print("BAT_INSTRUCTIONS loaded:", len(BAT_INSTRUCTIONS), "chars")

## Build the few-shot block

Turns each row of `few_shot_ex10.csv` (Nadia's human labels) into an example
`(post text) -> (JSON answer)` pair. These get prepended to the system prompt
so the model sees 10 worked examples before it classifies anything.

In [ ]:
def build_fewshot_block(fewshot_df: pd.DataFrame) -> str:
    """Turn labeled examples into worked (text -> JSON) pairs for the prompt."""
    blocks = []
    for i, row in fewshot_df.reset_index(drop=True).iterrows():
        ex = {
            "EX":  str(row.get("EX", "NO")).strip().upper(),
            "EMO": str(row.get("EMO", "NO")).strip().upper(),
            "COG": str(row.get("COG", "NO")).strip().upper(),
            "MD":  str(row.get("MD", "NO")).strip().upper(),
        }
        # Nadia's file has no reasoning text — use a short neutral placeholder
        # for the reasoning fields in the demonstration JSON.
        answer = {
            "EX": ex["EX"], "EMO": ex["EMO"], "COG": ex["COG"], "MD": ex["MD"],
            "EX_reasoning":  f"Human label: {ex['EX']}",
            "EMO_reasoning": f"Human label: {ex['EMO']}",
            "COG_reasoning": f"Human label: {ex['COG']}",
            "MD_reasoning":  f"Human label: {ex['MD']}",
        }
        text = str(row["text"]).strip()
        blocks.append(
            f"Example {i+1}:\n"
            f'Reddit post:\n"""{text}"""\n\n'
            f"JSON answer:\n{json.dumps(answer, indent=2)}"
        )
    return "\n\n".join(blocks)


fewshot_df = pd.read_csv(FEWSHOT_FILE)
print(f"Loaded {len(fewshot_df)} few-shot examples from {FEWSHOT_FILE}")

FEWSHOT_BLOCK = build_fewshot_block(fewshot_df)
print(f"Few-shot block built: {len(FEWSHOT_BLOCK)} chars")

POST_SYSTEM = (
    "You are annotating Reddit POSTS from cybersecurity communities for a burnout study.\n"
    + BAT_INSTRUCTIONS
    + "\n\n\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\n"
      "WORKED EXAMPLES (human-labeled ground truth, follow this pattern)\n"
      "\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\u2501\n\n"
    + FEWSHOT_BLOCK
    + "\n\nNow classify the new post the same way. JSON only, matching the schema above exactly."
)
print("POST_SYSTEM total length:", len(POST_SYSTEM), "chars")

## Call the UVA RC GenAI endpoint (Kimi K2.5)

This endpoint streams Server-Sent Events even when `stream: False` is requested
(known quirk from the Phase 2A HPC pipeline). This helper tries a plain JSON
parse first, and falls back to SSE line parsing if that fails, discarding any
`reasoning` deltas and keeping only `content`.

In [ ]:
def call_kimi(system: str, user: str, row_id: str, max_tokens: int = MAX_TOKENS,
              max_retries: int = 2) -> dict:
    """Call the UVA RC GenAI endpoint (Kimi K2.5) and return parsed JSON dict.
    Retries transient failures (timeouts, malformed/truncated JSON) up to
    max_retries times before giving up. Always returns a dict with '_error'
    and '_raw' keys on failure so the reason is inspectable, not just printed.
    """
    headers = {
        "Authorization": f"Bearer {UVARC_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "temperature": TEMPERATURE,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "stream": False,
    }

    last_error = None
    last_raw = ""

    for attempt in range(1, max_retries + 2):  # e.g. max_retries=2 -> up to 3 tries
        try:
            with httpx.Client(timeout=120.0) as client:
                resp = client.post(CHAT_URL, headers=headers, json=payload)
                if resp.status_code >= 400:
                    last_error = f"HTTP {resp.status_code}: {resp.text[:500]}"
                    last_raw = resp.text
                    print(f"    [!] attempt {attempt}: HTTP {resp.status_code} for {row_id} — {resp.text[:200]}")
                    if resp.status_code in (429, 500, 502, 503, 504):
                        time.sleep(2 * attempt)  # backoff, then retry
                        continue
                    break  # non-retryable HTTP error (e.g. 400 bad request)
                raw_body = resp.text

            # Try plain JSON first (non-streamed response)
            content = None
            try:
                data = json.loads(raw_body)
                content = data["choices"][0]["message"]["content"]
            except (json.JSONDecodeError, KeyError, IndexError, TypeError):
                # Fall back to SSE parsing: lines like "data: {...}", discard
                # delta.reasoning, keep delta.content only.
                parts = []
                for line in raw_body.splitlines():
                    line = line.strip()
                    if not line.startswith("data:"):
                        continue
                    chunk = line[len("data:"):].strip()
                    if chunk == "[DONE]" or not chunk:
                        continue
                    try:
                        obj = json.loads(chunk)
                    except json.JSONDecodeError:
                        continue
                    delta = obj.get("choices", [{}])[0].get("delta", {})
                    if "content" in delta and delta["content"]:
                        parts.append(delta["content"])
                    # delta.get("reasoning") intentionally discarded
                content = "".join(parts)

            if not content:
                last_error = f"Empty/unparseable response body"
                last_raw = raw_body[:1000]
                print(f"    [!] attempt {attempt}: {row_id} — empty content, raw: {raw_body[:200]}")
                time.sleep(1)
                continue

            raw = content.strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            s = raw.find("{"); e = raw.rfind("}") + 1
            if s != -1 and e > s:
                raw = raw[s:e]

            try:
                parsed = json.loads(raw)
                required = {"EX", "EMO", "COG", "MD"}
                if not required.issubset(parsed.keys()):
                    last_error = f"Missing keys in response: {required - parsed.keys()}"
                    last_raw = content[:1000]
                    print(f"    [!] attempt {attempt}: {row_id} — {last_error}")
                    time.sleep(1)
                    continue
                return parsed  # success
            except json.JSONDecodeError as je:
                last_error = f"JSON parse failed: {je}"
                last_raw = content[:1000]
                print(f"    [!] attempt {attempt}: {row_id} — JSON parse failed on: {content[:200]}")
                time.sleep(1)
                continue

        except Exception as ex:
            last_error = str(ex)
            print(f"    [!] attempt {attempt}: exception for {row_id}: {ex}")
            time.sleep(1)
            continue

    # All attempts exhausted
    return {"_error": last_error, "_raw": last_raw}


def bat_score(result: dict) -> int:
    return sum(1 for k in ["EX", "EMO", "COG", "MD"] if result.get(k) == "YES")


def empty_bat(error_msg: str = "", raw: str = "") -> dict:
    return {
        "EX": "ERROR", "EMO": "ERROR", "COG": "ERROR", "MD": "ERROR",
        "EX_reasoning":  f"API error: {error_msg}"[:500],
        "EMO_reasoning": f"API error: {error_msg}"[:500],
        "COG_reasoning": f"API error: {error_msg}"[:500],
        "MD_reasoning":  f"raw response (truncated): {raw}"[:500],
    }

## Quick connectivity test

Run this once before the full loop to confirm the endpoint, model name, and
API key all work. If this fails, fix it here before running the full 30-row loop.

In [ ]:
test_result = call_kimi(
    POST_SYSTEM,
    'Reddit post to annotate:\n"""I am so drained lately, work has been non-stop and I can\'t think straight anymore."""\n\nJSON only.',
    "connectivity_test",
)
print(json.dumps(test_result, indent=2))

### If you got a 400 error

The cell above now prints the server's actual response body — re-run it and read that message first, it usually says exactly what's wrong (unknown model name, missing/extra field, bad auth format, etc.).

If the body doesn't make it obvious, this cell tries to list models the endpoint actually recognizes, which is the most common cause of a 400 here.

In [ ]:
# Optional: list models this endpoint recognizes (helps if MODEL name is wrong)
try:
    with httpx.Client(timeout=30.0) as client:
        r = client.get(f"{BASE_URL}/models", headers={"Authorization": f"Bearer {UVARC_API_KEY}"})
        print(r.status_code)
        print(r.text[:2000])
except Exception as ex:
    print("Could not list models:", ex)

## Load the 30 posts to classify, check for existing progress

Re-running this notebook is safe: any `post_id` already present in
`few_shot_bat_classification_labeled.csv` is skipped.

In [ ]:
input_df = pd.read_csv(INPUT_FILE)
print(f"Posts to classify (input): {len(input_df)}")
print(input_df["post_id"].tolist())

processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    try:
        existing_df = pd.read_csv(OUTPUT_FILE)
        processed_ids = set(existing_df["post_id"].astype(str).tolist())
        print(f"Resuming: {len(processed_ids)} posts already classified, will skip those.")
    except pd.errors.EmptyDataError:
        print("Output file exists but is empty — starting fresh.")
else:
    print("No existing output — starting fresh.")

to_process_df = input_df[~input_df["post_id"].astype(str).isin(processed_ids)].copy()
print(f"Remaining to process: {len(to_process_df)}")

## Classify sequentially

n=30, single sequential loop — no threads, no batching. Each row is appended
to the output CSV immediately after it's classified, so partial progress is
never lost even if the notebook is interrupted.

In [ ]:
results = []

for i, (_, row) in enumerate(to_process_df.iterrows(), start=1):
    pid = str(row["post_id"])
    text = str(row["text"])
    preview = text[:70].replace("\n", " ")
    print(f"[{i}/{len(to_process_df)}] POST {pid}")
    print(f'  "{preview}..."')

    prompt = f'Reddit post to annotate:\n"""{text.strip()}"""\n\nJSON only.'
    result = call_kimi(POST_SYSTEM, prompt, pid)

    if "_error" in result:
        bat = empty_bat(error_msg=result.get("_error", ""), raw=result.get("_raw", ""))
        print(f"  -> FAILED after retries: {result.get('_error')}")
    else:
        bat = result

    score = bat_score(bat)
    flags = f"EX={bat.get('EX')} EMO={bat.get('EMO')} COG={bat.get('COG')} MD={bat.get('MD')}"
    print(f"  -> {flags} | bat_score={score}")

    row_out = {
        "row_type":      row.get("row_type", "post"),
        "post_id":       pid,
        "comment_id":    row.get("comment_id", ""),
        "text":          text,
        "EX":            bat.get("EX", "ERROR"),
        "EMO":           bat.get("EMO", "ERROR"),
        "COG":           bat.get("COG", "ERROR"),
        "MD":            bat.get("MD", "ERROR"),
        "bat_score":     score,
        "EX_reasoning":  bat.get("EX_reasoning", ""),
        "EMO_reasoning": bat.get("EMO_reasoning", ""),
        "COG_reasoning": bat.get("COG_reasoning", ""),
        "MD_reasoning":  bat.get("MD_reasoning", ""),
    }

    # Append immediately (n=30, no need to batch this)
    row_df = pd.DataFrame([row_out])
    file_is_new = not os.path.exists(OUTPUT_FILE) or os.path.getsize(OUTPUT_FILE) == 0
    row_df.to_csv(OUTPUT_FILE, mode="w" if file_is_new else "a", index=False, header=file_is_new)

    results.append(row_out)
    time.sleep(DELAY_SECONDS)

print(f"\nDone. {len(results)} new rows classified this run.")

## Summary

In [ ]:
final_df = pd.read_csv(OUTPUT_FILE)
print(f"Total posts in output: {len(final_df)}")
print(f"BAT score distribution: {final_df['bat_score'].value_counts().sort_index().to_dict()}")
print(f"Avg bat_score: {final_df['bat_score'].mean():.2f}")
for cat in ["EX", "EMO", "COG", "MD"]:
    yes_n = (final_df[cat] == "YES").sum()
    err_n = (final_df[cat] == "ERROR").sum()
    print(f"  {cat}: YES={yes_n}/{len(final_df)} ({yes_n/len(final_df)*100:.1f}%)  ERROR={err_n}")

print(f"\n\u2713 Results saved to -> {OUTPUT_FILE}")

## Retry failed (ERROR) rows only

If the summary above shows `ERROR` counts, this deletes just those rows from the output CSV and re-runs the loop, which will naturally pick them back up as \"remaining to process\" the next time you run the 'Classify sequentially' cell above. Run this, then scroll up and re-run the 'Classify sequentially' and 'Summary' cells.

In [ ]:
# Remove ERROR rows from the output so they get reclassified on next run
if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    n_before = len(df)
    error_mask = (df[["EX","EMO","COG","MD"]] == "ERROR").any(axis=1)
    n_errors = error_mask.sum()
    df_clean = df[~error_mask]
    df_clean.to_csv(OUTPUT_FILE, index=False)
    print(f"Removed {n_errors} ERROR row(s) out of {n_before}. "
          f"{len(df_clean)} good rows kept in {OUTPUT_FILE}.")
    print("Now scroll up and re-run the 'Classify sequentially' cell to retry them.")
else:
    print("No output file found yet.")